Visualisierungen der einzelnen Klassen

Anzahl Geschosse

In [1]:
import geopandas as gpd
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file("av_projected_buildings_final.gpkg", layer="LCSFPROJ_final", engine="pyogrio")

# ── KEY FIX: fill NaN BEFORE value_counts ────────────────────────────────────
col = "Anzahl Geschosse"
series = gdf[col].fillna("Keine Angabe").astype(str)

counts = series.value_counts().reset_index()
counts.columns = ["Geschosse", "Anzahl"]

# Sort: numeric first, "Keine Angabe" last
def sort_key(val):
    try:
        return (0, float(val))
    except:
        return (1, val)

counts = counts.sort_values("Geschosse", key=lambda x: x.map(sort_key)).reset_index(drop=True)

# ── CHART ─────────────────────────────────────────────────────────────────────
fig = go.Figure(go.Bar(
    x=counts["Geschosse"],
    y=counts["Anzahl"],
    # text=counts["Anzahl"].apply(lambda v: f"{v:,}"),
    textposition="outside",
))

fig.update_traces(cliponaxis=False)
fig.update_layout(
    title={
        "text": "Anzahl Geschosse – Verteilung CH"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                "Quelle: AV/GWR | inkl. fehlende Werte</span>"
    }
)
fig.update_xaxes(title_text="Geschosse", type="category")
fig.update_yaxes(title_text="Anzahl Gebäude")

fig.write_image("anzahl_geschosse.png")
with open("anzahl_geschosse.png.meta.json", "w") as f:
    json.dump({
        "caption": "Verteilung Anzahl Geschosse (inkl. fehlende Werte)",
        "description": "Bar chart of floor count distribution including missing values"
    }, f)

print(counts.to_string(index=False))


   Geschosse  Anzahl
         1.0    6087
         2.0    8507
         3.0    9525
         4.0    4274
         5.0    1763
         6.0     926
         7.0     488
         8.0     269
         9.0     182
        10.0      93
        11.0      29
        12.0      10
        13.0      12
        14.0      10
        15.0       4
        16.0       2
        17.0       2
        18.0       2
        19.0       5
        20.0       5
        21.0       3
        23.0       4
        24.0       3
        25.0       4
        26.0       1
        27.0       2
        28.0       3
        29.0       1
        30.0       1
        33.0       2
Keine Angabe   15029


Anzahl EGIDS

In [2]:
import geopandas as gpd
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file("av_projected_buildings_final.gpkg", layer="LCSFPROJ_final", engine="pyogrio")

# ── COUNT EGID COVERAGE ───────────────────────────────────────────────────────
total = len(gdf)
has_egid = gdf["GWR_EGID"].notna() & (gdf["GWR_EGID"] != 0)
count_with    = has_egid.sum()
count_without = (~has_egid).sum()
pct_with    = count_with / total * 100
pct_without = count_without / total * 100

print(f"Total features:    {total:,}")
print(f"Mit EGID:          {count_with:,}  ({pct_with:.1f}%)")
print(f"Ohne EGID:         {count_without:,} ({pct_without:.1f}%)")

# ── PIE CHART ─────────────────────────────────────────────────────────────────
labels = ["Mit EGID", "Ohne EGID"]
values = [count_with, count_without]

fig = go.Figure(go.Pie(
    labels=labels,
    values=values,
    textinfo="label+percent+value",
    texttemplate="%{label}<br>%{percent}<br>%{value:,}",
    hole=0.3,
))

fig.update_layout(
    title={
        "text": "EGID Abdeckung – Projizierte Gebäude CH"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                f"Quelle: AV/GWR | Total: {total:,} Gebäude</span>"
    },
    uniformtext_minsize=14,
    uniformtext_mode="hide",
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="right", x=1.1)
)

fig.write_image("egid_coverage.png")
with open("egid_coverage.png.meta.json", "w") as f:
    json.dump({
        "caption": "EGID Abdeckung projizierte Gebäude Schweiz",
        "description": "Pie chart showing how many projected buildings have a GWR_EGID value vs missing"
    }, f)


Total features:    47,248
Mit EGID:          37,589  (79.6%)
Ohne EGID:         9,659 (20.4%)


In [3]:
import geopandas as gpd
import plotly.graph_objects as go
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file("av_projected_buildings_final.gpkg", layer="LCSFPROJ_final", engine="pyogrio")

# ── COUNT ─────────────────────────────────────────────────────────────────────
total = len(gdf)
has_val = gdf["Anzahl Geschosse"].notna()
count_with    = has_val.sum()
count_without = (~has_val).sum()
pct_with    = count_with / total * 100
pct_without = count_without / total * 100

print(f"Total:          {total:,}")
print(f"Mit Wert:       {count_with:,}  ({pct_with:.1f}%)")
print(f"Keine Angabe:   {count_without:,} ({pct_without:.1f}%)")

# ── PIE CHART ─────────────────────────────────────────────────────────────────
fig = go.Figure(go.Pie(
    labels=["Mit Wert", "Keine Angabe"],
    values=[count_with, count_without],
    textinfo="label+percent",
    texttemplate="%{label}<br>%{percent}<br>%{value:,}",
    hole=0.3,
))

fig.update_layout(
    title={
        "text": "Anzahl Geschosse – Datenvollständigkeit CH"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                f"Quelle: AV/GWR | Total: {total:,} Gebäude</span>"
    },
    uniformtext_minsize=14,
    uniformtext_mode="hide",
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="right", x=1.1)
)

fig.write_image("geschosse_coverage.png")
with open("geschosse_coverage.png.meta.json", "w") as f:
    json.dump({
        "caption": "Datenvollständigkeit Anzahl Geschosse",
        "description": "Pie chart showing share of projected buildings with and without a floor count value"
    }, f)


Total:          47,248
Mit Wert:       32,219  (68.2%)
Keine Angabe:   15,029 (31.8%)


has EGID Value by Canton


In [4]:
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file("av_projected_buildings_final.gpkg", layer="LCSFPROJ_final", engine="pyogrio")
gdf["GWR_EGID"] = pd.to_numeric(gdf["GWR_EGID"], errors="coerce")

# ── COUNT PER CANTON ──────────────────────────────────────────────────────────
canton_stats = gdf.groupby("Kanton").apply(
    lambda x: pd.Series({
        "Mit EGID":    (x["GWR_EGID"].notna() & (x["GWR_EGID"] != 0)).sum(),
        "Ohne EGID":   (x["GWR_EGID"].isna()  | (x["GWR_EGID"] == 0)).sum(),
        "Total":       len(x),
    })
).reset_index()

canton_stats["% Mit EGID"] = (canton_stats["Mit EGID"] / canton_stats["Total"] * 100).round(1)
canton_stats = canton_stats.sort_values("% Mit EGID", ascending=False)

print(canton_stats.to_string(index=False))

# ── GROUPED BAR CHART ─────────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Bar(
    name="Mit EGID",
    x=canton_stats["Kanton"],
    y=canton_stats["Mit EGID"],
    text=canton_stats["Mit EGID"].apply(lambda v: f"{v:,}"),
    textposition="outside",
))

fig.add_trace(go.Bar(
    name="Ohne EGID",
    x=canton_stats["Kanton"],
    y=canton_stats["Ohne EGID"],
    text=canton_stats["Ohne EGID"].apply(lambda v: f"{v:,}"),
    textposition="outside",
))

fig.update_traces(cliponaxis=False)
fig.update_layout(
    barmode="group",
    title={
        "text": "EGID Abdeckung pro Kanton"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                "Quelle: AV/GWR | sortiert nach % Mit EGID</span>"
    },
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5)
)
fig.update_xaxes(title_text="Kanton")
fig.update_yaxes(title_text="Anzahl Gebäude")

fig.write_image("egid_coverage_kanton.png")
with open("egid_coverage_kanton.png.meta.json", "w") as f:
    json.dump({
        "caption": "EGID Abdeckung pro Kanton (Mit vs. Ohne)",
        "description": "Grouped bar chart showing buildings with and without EGID per Swiss canton, sorted by coverage rate"
    }, f)

# ── ALSO SAVE STATS AS CSV ────────────────────────────────────────────────────
canton_stats.to_csv("egid_coverage_kanton.csv", index=False)
print("\nSaved → egid_coverage_kanton.png + egid_coverage_kanton.csv")


Kanton  Mit EGID  Ohne EGID  Total  % Mit EGID
    GE      5798          0   5798       100.0
    BS       234          1    235        99.6
    BL      1515         22   1537        98.6
    GL       238          9    247        96.4
    BE      3802        309   4111        92.5
    TI      2449        244   2693        90.9
    SZ      1241        179   1420        87.4
    ZH      5373       1244   6617        81.2
    AG      4646       1117   5763        80.6
    SG      3354       1272   4626        72.5
    FR      3123       1323   4446        70.2
    SO       696        328   1024        68.0
    TG      1633        899   2532        64.5
    ZG       507        279    786        64.5
    UR       163         90    253        64.4
    VS       415        231    646        64.2
    GR      1633        938   2571        63.5
    AR       354        466    820        43.2
    AI       182        304    486        37.4
    SH       233        404    637        36.6

Saved → egid